In [9]:
# Ensure project src is on sys.path for imports
import os, sys
src_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print(f"Added to sys.path: {src_dir}")

Added to sys.path: c:\Users\david\Documents\GitHub\slab_net_zero\src


In [10]:
from typing import Tuple
import numpy as np

from slab_net_zero.motion_planning.ik_offline.geometry import Plane
from slab_net_zero.motion_planning.ik_offline.trans import (
    world_to_plane,
    plane_to_world,
    from_plane_to_plane,
    apply_T_to_plane,
)
from slab_net_zero.motion_planning.ik_offline.ik_offline import (
    ik_with_tool,
    print_matrix,
    print_plane,
)

print("Imports ok.")


Inverse Kinematics Solutions:
[2.121796, -0.404935, 2.300289, 1.647714, 1.056038, -0.206041]
[2.121796, 1.525000, -2.300289, -1.964830, 1.056038, -0.206041]
[2.121796, -0.870919, 2.263498, -0.991105, -1.056038, 2.935551]
[2.121796, 1.039412, -2.263498, 1.625561, -1.056038, 2.935551]
[-0.394834, 2.086848, 2.192272, 1.249410, 2.622006, 2.457034]
[-0.394834, -2.327387, -2.192272, -2.518181, 2.622006, 2.457034]
[-0.394834, 1.599835, 2.378749, -1.591647, -2.622006, -0.684559]
[-0.394834, -2.716563, -2.378749, 1.199064, -2.622006, -0.684559]
Imports ok.


## Define planes
World frame and two example planes (from `test_trans.py`) to exercise transforms.

In [11]:
# World XY frame
world_xy = Plane(origin=(0.0, 0.0, 0.0), xaxis=(1.0, 0.0, 0.0), yaxis=(0.0, 1.0, 0.0))

# Planes from test_trans.py
plane_from = Plane(
    origin=(-0.731, 0.695, 0.528),
    xaxis=(0.248, -0.723, -0.645),
    yaxis=(0.616, 0.632, -0.471),
)
plane_to = Plane(
    origin=(0.912, 0.896, -0.887),
    xaxis=(-0.784, -0.101, 0.612),
    yaxis=(0.322, -0.909, 0.263),
)

print_plane(world_xy, "world_xy")
print_plane(plane_from, "plane_from")
print_plane(plane_to, "plane_to")

world_xy:
  origin: (0.0, 0.0, 0.0)
  xaxis : (1.0, 0.0, 0.0)
  yaxis : (0.0, 1.0, 0.0)
  zaxis : (0.0, 0.0, 1.0)
plane_from:
  origin: (-0.731, 0.695, 0.528)
  xaxis : (0.248, -0.723, -0.645)
  yaxis : (0.616, 0.632, -0.471)
  zaxis : (0.748173, -0.280512, 0.602104)
plane_to:
  origin: (0.912, 0.896, -0.887)
  xaxis : (-0.784, -0.101, 0.612)
  yaxis : (0.322, -0.909, 0.263)
  zaxis : (0.529745, 0.403256, 0.745178)


In [19]:
from compas.geometry import Frame, Transformation
frame_from = Frame(point=plane_from.origin, xaxis=plane_from.xaxis, yaxis=plane_from.yaxis)
frame_to = Frame(point=plane_to.origin, xaxis=plane_to.xaxis, yaxis=plane_to.yaxis)

T = Transformation.from_frame_to_frame(frame_from, frame_to)
print_matrix(T)

  0.400522   0.621882   0.672938   0.417262
 -0.283323  -0.614375   0.736391   0.727067
  0.871384  -0.485599  -0.069877   0.124369
  0.000000   0.000000   0.000000   1.000000


## Compose transform and apply
Use `from_plane_to_plane` and `apply_T_to_plane`, mirroring `test_trans.py` expectations.

In [13]:
T = from_plane_to_plane(plane_from, plane_to)
print("Transformation (plane_from -> plane_to):")
print_matrix(T)

transformed = apply_T_to_plane(T, plane_from)
print_plane(transformed, "transformed (should match plane_to)")

print("origin matches:", np.allclose(transformed.origin, plane_to.origin, atol=1e-3))
print("xaxis matches:", np.allclose(transformed.xaxis, plane_to.xaxis, atol=1e-3))
print("yaxis matches:", np.allclose(transformed.yaxis, plane_to.yaxis, atol=1e-3))

Transformation (plane_from -> plane_to):
  0.400339   0.621812   0.673133   0.417075
 -0.283258  -0.614640   0.736187   0.727406
  0.871480  -0.485406  -0.069905   0.124319
  0.000000   0.000000   0.000000   1.000000
transformed (should match plane_to):
  origin: (np.float64(0.912), np.float64(0.896), np.float64(-0.887))
  xaxis : (np.float64(-0.784355), np.float64(-0.100691), np.float64(0.612085))
  yaxis : (np.float64(0.322432), np.float64(-0.909356), np.float64(0.262886))
  zaxis : (np.float64(0.530133), np.float64(0.403552), np.float64(0.745724))
origin matches: True
xaxis matches: True
yaxis matches: True


## Inverse kinematics with tool
Use `ik_with_tool` to compute UR joint solutions for a target flange plane.

In [14]:
target_plane = Plane(
    origin=(0.5, 0.0, 0.5),
    xaxis=(1.0, 0.0, 0.0),
    yaxis=(0.0, 1.0, 0.0),
)

ik_solutions = ik_with_tool(target_plane)
print(f"Found {len(ik_solutions)} IK solution(s). Showing first 4:")
for i, sol in enumerate(ik_solutions[:4]):
    print(f"#{i+1}: [" + ", ".join(f"{a:+.6f}" for a in sol) + "]")

Found 8 IK solution(s). Showing first 4:
#1: [+2.121796, -0.404935, +2.300289, +1.647714, +1.056038, -0.206041]
#2: [+2.121796, +1.525000, -2.300289, -1.964830, +1.056038, -0.206041]
#3: [+2.121796, -0.870919, +2.263498, -0.991105, -1.056038, +2.935551]
#4: [+2.121796, +1.039412, -2.263498, +1.625561, -1.056038, +2.935551]


## Notes
- Transforms are the same as validated in `tests/test_trans.py`.
- `ik_with_tool` uses the TCP offset defined in `ik_offline.py`; adjust there if your tool changes.